# Prompt 9 — Resource Dependencies and Lifecycle
## Terraform Associate (004) Exam Study Guide

**Exam Objective:** Understand how Terraform orders resource operations, manages dependencies, controls lifecycle behaviour, and handles multi-provider configurations.

---

## Table of Contents
1. [Implicit Dependencies](#1-implicit-dependencies)
2. [Explicit Dependencies — `depends_on`](#2-explicit-dependencies--depends_on)
3. [Dependency Graph (DAG)](#3-dependency-graph-dag)
4. [Parallel Execution](#4-parallel-execution)
5. [The `-target` Flag](#5-the--target-flag)
6. [`terraform graph` Command](#6-terraform-graph-command)
7. [Resource Lifecycle Meta-Arguments](#7-resource-lifecycle-meta-arguments)
8. [`provider` Meta-Argument and Aliases](#8-provider-meta-argument-and-aliases)
9. [`moved` Block](#9-moved-block)
10. [`removed` Block](#10-removed-block)
11. [Exam-Style Questions](#11-exam-style-questions)
12. [Real-World Scenarios](#12-real-world-scenarios)
13. [Quick-Reference Cheat Sheet](#13-quick-reference-cheat-sheet)

---
## 1. Implicit Dependencies

Terraform **automatically** infers the order in which resources must be created or destroyed by scanning for attribute references between resources.

### How It Works
When resource **B** references an attribute of resource **A** (e.g., `aws_vpc.main.id`), Terraform adds a directed edge **A → B** in the dependency graph.  
This means **A is created before B**, and **B is destroyed before A**.

### Reference Syntax That Creates an Edge

| Expression | What It References |
|---|---|
| `aws_vpc.main.id` | `id` attribute of `aws_vpc.main` |
| `aws_subnet.web.cidr_block` | `cidr_block` of `aws_subnet.web` |
| `module.network.vpc_id` | output `vpc_id` from module `network` |
| `var.region` | input variable — no dependency edge |
| `local.tags` | local value — no dependency edge |

Only **resource and module** references create edges; variables and locals do not.

In [ ]:
# ── IMPLICIT DEPENDENCY EXAMPLE ──────────────────────────────────────────────
# The subnet references aws_vpc.main.id, so Terraform creates the VPC first.

implicit_dependency_hcl = '''
resource "aws_vpc" "main" {
  cidr_block = "10.0.0.0/16"

  tags = { Name = "main-vpc" }
}

resource "aws_subnet" "web" {
  # Referencing aws_vpc.main.id creates an implicit dependency.
  # Terraform will create aws_vpc.main BEFORE aws_subnet.web.
  vpc_id            = aws_vpc.main.id
  cidr_block        = "10.0.1.0/24"
  availability_zone = "us-east-1a"

  tags = { Name = "web-subnet" }
}

resource "aws_security_group" "web_sg" {
  # Also depends on aws_vpc.main implicitly.
  vpc_id      = aws_vpc.main.id
  name        = "web-sg"
  description = "Allow HTTP"

  ingress {
    from_port   = 80
    to_port     = 80
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }
}
'''

print("Execution order Terraform infers:")
print("  1. aws_vpc.main         (no dependencies)")
print("  2. aws_subnet.web       (depends on aws_vpc.main.id)")
print("     aws_security_group.web_sg (depends on aws_vpc.main.id — parallel with subnet)")
print()
print(implicit_dependency_hcl)

---
## 2. Explicit Dependencies — `depends_on`

Use `depends_on` when a dependency **exists but is not visible in the configuration** — i.e., Terraform cannot detect it from attribute references alone.

### When to Use
- Resource B depends on a **side effect** of resource A (e.g., an IAM policy must be attached before an EC2 instance boots and tries to call an AWS API)
- Resource B depends on an action taken by a **null_resource** or **local-exec provisioner**
- Module A must complete before module B starts, but B does not reference any of A's outputs

### Syntax
```hcl
resource "aws_instance" "web" {
  # ...
  depends_on = [aws_iam_role_policy_attachment.web_policy]
}
```

> **Exam tip:** `depends_on` accepts a **list of resource or module references** — not attribute paths.  
> Use `aws_iam_role_policy_attachment.web_policy`, not `aws_iam_role_policy_attachment.web_policy.id`.

In [ ]:
# ── EXPLICIT DEPENDENCY EXAMPLE ───────────────────────────────────────────────
# The EC2 instance must launch AFTER the IAM policy is attached.
# Terraform cannot detect this from attribute references, so we use depends_on.

depends_on_hcl = '''
resource "aws_iam_role" "ec2_role" {
  name               = "ec2-s3-access"
  assume_role_policy = data.aws_iam_policy_document.ec2_assume.json
}

resource "aws_iam_role_policy_attachment" "s3_read" {
  role       = aws_iam_role.ec2_role.name
  policy_arn = "arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess"
}

resource "aws_instance" "app" {
  ami           = "ami-0abcdef1234567890"
  instance_type = "t3.micro"
  iam_instance_profile = aws_iam_instance_profile.ec2_profile.name

  # The instance startup script calls S3. The IAM policy must be fully
  # attached before the instance boots — but this isn't visible from
  # attribute references alone, so depends_on makes the edge explicit.
  depends_on = [aws_iam_role_policy_attachment.s3_read]
}
'''

print("Without depends_on: Terraform might create the instance before the")
print("  policy is attached, causing a permissions error at boot.")
print()
print("With depends_on: Terraform guarantees the policy attachment completes")
print("  before the instance is created.")
print()
print(depends_on_hcl)

---
## 3. Dependency Graph (DAG)

Terraform models all resources and their relationships as a **Directed Acyclic Graph (DAG)**.

### Key Properties

| Property | Meaning |
|---|---|
| **Directed** | Edges have direction: A must exist before B |
| **Acyclic** | No circular dependencies — if A depends on B and B depends on A, Terraform errors |
| **Graph** | Multiple resources can converge on a single dependency |

### How Operations Use the Graph

| Operation | Graph Direction |
|---|---|
| `terraform apply` (create) | Walk from root toward leaves — parents before children |
| `terraform destroy` | Walk in **reverse** — children before parents |
| `terraform plan` | Full graph traversal to determine diffs |

### Visualising the Graph

```bash
# Generate DOT format output
terraform graph

# Pipe to Graphviz to render a PNG
terraform graph | dot -Tpng > graph.png

# Or paste the DOT output into https://dreampuf.github.io/GraphvizOnline/
```

> **Exam tip:** `terraform graph` outputs **DOT format** (a graph description language).  
> You need external tooling (Graphviz) to render it visually.

---
## 4. Parallel Execution

Terraform **maximises parallelism** by applying independent resources simultaneously.

### How It Works
- Resources with **no dependency relationship** are created/destroyed in parallel
- Default concurrency: **10 operations at once** (controlled by `-parallelism=N` flag)
- Once a resource's dependencies are all complete, it becomes eligible to start immediately

### Example Graph

```
aws_vpc.main
  ├── aws_subnet.web        ─┐
  ├── aws_subnet.app        ─┤  These 3 subnets have no
  └── aws_subnet.db         ─┘  dependency on each other
                                  → created in parallel

aws_instance.web
  depends on: aws_subnet.web, aws_security_group.web_sg
  → starts only after BOTH dependencies complete
```

### `-parallelism` Flag

```bash
# Reduce concurrency to 5 (useful in rate-limited environments)
terraform apply -parallelism=5

# Disable parallelism entirely (serial execution)
terraform apply -parallelism=1
```

---
## 5. The `-target` Flag

The `-target` flag restricts a `plan`, `apply`, or `destroy` operation to **one or more specific resources**, ignoring the rest of the configuration.

### Syntax

```bash
# Target a single resource
terraform apply -target=aws_instance.web

# Target a resource by full address (with module path)
terraform apply -target=module.network.aws_vpc.main

# Target multiple resources
terraform apply -target=aws_instance.web -target=aws_security_group.web_sg

# Target a specific instance of a count resource
terraform apply -target='aws_instance.web[0]'

# Target destroy
terraform destroy -target=aws_instance.web
```

### Valid Use Cases

| Use Case | Why |
|---|---|
| Unblock a failed apply on one resource | Fix the specific resource without touching others |
| Bootstrap a dependency cycle | Create one resource first to break a chicken-and-egg problem |
| Iterative development | Quickly test one resource in isolation |

### ⚠️ Cautions

> **Exam warning:** `-target` is **not for routine use** — it is a troubleshooting escape hatch.

- Targeted applies can leave state **inconsistent** with configuration — dependent resources may be out of sync
- After a targeted apply, you must eventually run a full `terraform apply` to reconcile everything
- Terraform will warn you when you use `-target` that the resulting state may be incomplete

---
## 6. `terraform graph` Command

Outputs a **DOT-language** representation of the resource dependency graph for the current configuration.

### Key Facts for the Exam

| Fact | Detail |
|---|---|
| Output format | DOT (Graphviz graph description language) |
| Rendering tool | Graphviz (`dot` CLI) or online viewer |
| Graph type flag | `-type=plan`, `-type=apply`, `-type=destroy`, `-type=plan-destroy` |
| Use case | Debugging dependency ordering, documentation, understanding module relationships |

### Example Commands

```bash
# Print DOT graph to stdout
terraform graph

# Render as PNG using Graphviz
terraform graph | dot -Tpng -o dependency-graph.png

# Show destroy graph
terraform graph -type=destroy
```

### Sample DOT Output (abbreviated)

```dot
digraph {
  compound = "true"
  newrank  = "true"
  subgraph "root" {
    "[root] aws_subnet.web (expand)" -> "[root] aws_vpc.main (expand)"
    "[root] aws_instance.web (expand)" -> "[root] aws_subnet.web (expand)"
    "[root] aws_instance.web (expand)" -> "[root] aws_security_group.web_sg (expand)"
  }
}
```

Each arrow (`->`) means "depends on" — the left side requires the right side to exist first.

---
## 7. Resource Lifecycle Meta-Arguments

The `lifecycle` block is nested inside a `resource` block and controls **how Terraform handles resource replacement and drift**.

### 7.1 Default Lifecycle (No `lifecycle` Block)

When a resource must be replaced (an immutable attribute changed):
```
1. Plan: Terraform shows -/+ (destroy then create)
2. Apply: destroy the old resource
3. Apply: create the new resource
```

This causes **downtime** for resources that other resources depend on.

### 7.2 `create_before_destroy = true`

Reverses the replace order: create the new resource **first**, then destroy the old one.

**Required for:** Load balancer target groups, TLS certificates, security groups — anything that must exist before the thing referencing it is updated.

```hcl
resource "aws_instance" "web" {
  ami           = var.ami_id
  instance_type = "t3.micro"

  lifecycle {
    create_before_destroy = true
  }
}
```

> **Exam tip:** If resource A sets `create_before_destroy = true` and resource B **depends on A**, Terraform propagates `create_before_destroy` to B automatically.

### 7.3 `prevent_destroy = true`

Terraform will **refuse to plan** any operation that would destroy this resource — the plan errors immediately.

**Use for:** Production databases, S3 buckets with data, any resource where accidental deletion would be catastrophic.

```hcl
resource "aws_db_instance" "prod" {
  # ...
  lifecycle {
    prevent_destroy = true
  }
}
```

> **Exam warning:** `prevent_destroy = true` does **not** protect against `terraform destroy -target` if the lifecycle block is removed before running destroy. The check happens at plan time — if the block is gone, the protection is gone.

### 7.4 `ignore_changes = [attribute_name]`

Terraform will **not compute a diff** for the listed attributes — changes made outside Terraform to those attributes will not trigger an update plan.

```hcl
resource "aws_instance" "web" {
  ami           = var.ami_id
  instance_type = "t3.micro"

  lifecycle {
    # Ignore changes to tags made by auto-tagging tools
    ignore_changes = [tags]

    # Ignore the AMI ID — auto-healing processes may update it out of band
    # ignore_changes = [ami]

    # Ignore ALL attributes (use sparingly — Terraform stops managing this resource)
    # ignore_changes = all
  }
}
```

> **Exam tip:** `ignore_changes = all` is a valid value — it tells Terraform to never update the resource after creation.

### 7.5 `replace_triggered_by`

Forces **replacement** of this resource whenever the referenced resource or attribute changes — even if none of this resource's own attributes change.

```hcl
resource "aws_launch_template" "app" {
  name_prefix   = "app-"
  image_id      = var.ami_id
  instance_type = "t3.micro"
}

resource "aws_autoscaling_group" "app" {
  # ...
  launch_template {
    id      = aws_launch_template.app.id
    version = aws_launch_template.app.latest_version
  }

  lifecycle {
    # Replace the ASG whenever the launch template changes
    replace_triggered_by = [aws_launch_template.app]
  }
}
```

### Summary Table

| Argument | Effect | Common Use Case |
|---|---|---|
| `create_before_destroy = true` | New resource before old is deleted | Zero-downtime replacement |
| `prevent_destroy = true` | Plan errors if destroy would occur | Protect production DBs |
| `ignore_changes = [attr]` | Skip drift detection for attribute | Tags managed externally |
| `ignore_changes = all` | Never update after creation | Externally managed resources |
| `replace_triggered_by = [ref]` | Force replace when reference changes | ASG + launch template coupling |

In [ ]:
# ── LIFECYCLE EXAMPLES ────────────────────────────────────────────────────────

lifecycle_hcl = '''
# ── create_before_destroy: zero-downtime TLS certificate replacement ──
resource "aws_acm_certificate" "api" {
  domain_name       = "api.example.com"
  validation_method = "DNS"

  lifecycle {
    # ACM certs cannot be deleted while in use.
    # Create the new cert first, update the listener, then delete the old cert.
    create_before_destroy = true
  }
}

# ── prevent_destroy: protect the production database ─────────────────────────
resource "aws_db_instance" "prod" {
  identifier        = "prod-postgres"
  engine            = "postgres"
  instance_class    = "db.t3.medium"
  allocated_storage = 100

  lifecycle {
    prevent_destroy = true
  }
}

# ── ignore_changes: tags managed by an external compliance tool ───────────────
resource "aws_instance" "app" {
  ami           = "ami-0abcdef1234567890"
  instance_type = "t3.small"

  tags = {
    Name = "app-server"
  }

  lifecycle {
    # The compliance tool writes additional tags at runtime.
    # Ignore drift in tags so Terraform does not remove them.
    ignore_changes = [tags]
  }
}

# ── replace_triggered_by: force ASG replacement when launch template changes ──
resource "aws_launch_template" "app" {
  name_prefix   = "app-"
  image_id      = var.ami_id
  instance_type = var.instance_type
}

resource "aws_autoscaling_group" "app" {
  name               = "app-asg"
  min_size           = 2
  max_size           = 10
  desired_capacity   = 2
  vpc_zone_identifier = [aws_subnet.app.id]

  launch_template {
    id      = aws_launch_template.app.id
    version = "$Latest"
  }

  lifecycle {
    replace_triggered_by  = [aws_launch_template.app]
    create_before_destroy = true
  }
}
'''

print(lifecycle_hcl)

---
## 8. `provider` Meta-Argument and Aliases

### Problem: Multiple Configurations of the Same Provider

By default, each provider type has **one configuration** in a workspace. But some use cases require **multiple** — e.g., deploying resources to two AWS regions simultaneously.

### Solution: Provider Aliases

Give each provider block a unique `alias` label, then use the `provider` meta-argument in resources to select which configuration to use.

### Rules

| Rule | Detail |
|---|---|
| Default provider | Provider block **without** `alias` — used by all resources by default |
| Aliased provider | Provider block **with** `alias` — must be explicitly selected |
| `provider =` syntax | `provider = <provider_type>.<alias>` inside the resource block |
| Not inheritable | Resources inside a module don't automatically inherit the caller's provider alias |

In [ ]:
# ── PROVIDER ALIAS EXAMPLE ────────────────────────────────────────────────────
# Deploy S3 buckets in two AWS regions using aliased providers.

provider_alias_hcl = '''
# ── providers.tf ─────────────────────────────────────────────────────────────

# Default provider — used by resources that do NOT specify provider =
provider "aws" {
  region = "us-east-1"
}

# Aliased provider — used only by resources that specify provider = aws.west
provider "aws" {
  alias  = "west"
  region = "us-west-2"
}

# ── main.tf ──────────────────────────────────────────────────────────────────

# Uses the default provider (us-east-1)
resource "aws_s3_bucket" "east_bucket" {
  bucket = "my-app-data-east"
}

# Explicitly selects the aliased provider (us-west-2)
resource "aws_s3_bucket" "west_bucket" {
  bucket = "my-app-data-west"

  provider = aws.west   # <── provider meta-argument
}

# ── Passing aliases into child modules ───────────────────────────────────────
# When calling a module, pass the aliased provider configuration explicitly.
# The module must declare a required_providers block to accept it.

module "disaster_recovery" {
  source = "./modules/dr-replication"

  providers = {
    aws         = aws          # default (us-east-1)
    aws.replica = aws.west     # aliased (us-west-2)
  }
}
'''

print("Key points:")
print("  - The default provider block (no alias) is used unless provider = is set")
print("  - Aliased providers require provider = <type>.<alias> in each resource")
print("  - Modules receive aliased providers via the providers = { } argument")
print()
print(provider_alias_hcl)

---
## 9. `moved` Block

The `moved` block lets you **rename or move** a resource in configuration without destroying and recreating it.  
Introduced in Terraform 1.1.

### When to Use
- Renaming a resource (`aws_instance.old_name` → `aws_instance.new_name`)
- Moving a resource into a module (`aws_vpc.main` → `module.network.aws_vpc.main`)
- Moving a resource out of a module
- Refactoring `count` to `for_each` (moving indexed instances to named instances)

### Syntax

```hcl
moved {
  from = aws_instance.old_name
  to   = aws_instance.new_name
}
```

### How It Works
1. Terraform detects the `moved` block during `terraform plan`
2. It updates the state file to point the old address to the new address
3. **No destroy/create** — the real infrastructure is untouched
4. After the team runs `terraform apply`, the `moved` block can be removed from configuration

> **Exam tip:** `moved` blocks are **idempotent** — leaving them in configuration after apply is safe but unnecessary.

In [ ]:
# ── MOVED BLOCK EXAMPLES ──────────────────────────────────────────────────────

moved_hcl = '''
# ── Example 1: Rename a resource ─────────────────────────────────────────────
# Before: resource "aws_instance" "server" { ... }
# After rename in config: resource "aws_instance" "web" { ... }

moved {
  from = aws_instance.server   # old address in state
  to   = aws_instance.web      # new address in config
}

# ── Example 2: Move resource into a module ────────────────────────────────────
# Before: resource "aws_vpc" "main" was in root module
# After: moved into module.network

moved {
  from = aws_vpc.main
  to   = module.network.aws_vpc.main
}

# ── Example 3: Migrate from count to for_each ─────────────────────────────────
# Before: resource "aws_subnet" "private" (count = 3), instances [0], [1], [2]
# After:  resource "aws_subnet" "private" (for_each over a map keyed by AZ name)

moved {
  from = aws_subnet.private[0]
  to   = aws_subnet.private["us-east-1a"]
}

moved {
  from = aws_subnet.private[1]
  to   = aws_subnet.private["us-east-1b"]
}

moved {
  from = aws_subnet.private[2]
  to   = aws_subnet.private["us-east-1c"]
}
'''

print(moved_hcl)

---
## 10. `removed` Block

The `removed` block tells Terraform to **stop managing a resource** without destroying the actual infrastructure.  
Introduced in Terraform 1.7.

### Use Cases
- A resource was created by Terraform but should now be managed manually
- Transferring ownership of infrastructure to another workspace
- Removing a resource from state because it was already manually deleted

### Syntax

```hcl
removed {
  from = aws_instance.legacy

  lifecycle {
    destroy = false   # false = remove from state only, do NOT destroy the real resource
    # destroy = true  # true  = destroy the real resource AND remove from state (default)
  }
}
```

> **Exam tip:** The key difference:
> - `destroy = false` → remove from **state only** (infrastructure untouched)
> - `destroy = true` → destroy **both** infrastructure and state entry

### Comparison: `moved` vs `removed`

| Block | Infrastructure | State | Use Case |
|---|---|---|---|
| `moved` | Unchanged | Address updated | Rename / move to module |
| `removed` (destroy=false) | Unchanged | Entry deleted | Stop managing, keep resource |
| `removed` (destroy=true) | **Deleted** | Entry deleted | Destroy and forget |

---
## 11. Exam-Style Questions

Test yourself — click **Answer** to reveal each explanation.

### Question 1

A Terraform configuration has an EC2 instance that depends on an IAM role policy attachment. The instance's startup script calls an AWS API that requires the policy. The instance references the IAM instance profile, but the policy attachment is only linked to the IAM role — not directly referenced by the instance. What is the correct approach?

**A.** Terraform detects the dependency automatically through the instance profile reference, so no additional configuration is needed.

**B.** Add `depends_on = [aws_iam_role_policy_attachment.the_policy]` to the EC2 instance resource.

**C.** Add `create_before_destroy = true` to the EC2 instance lifecycle block.

**D.** Use `ignore_changes = [iam_instance_profile]` to prevent Terraform from managing the IAM relationship.

<details><summary>Answer</summary>

**Answer: B**

The instance references `aws_iam_instance_profile.the_profile.name` — not `aws_iam_role_policy_attachment.the_policy`. Terraform therefore does not infer a dependency between the instance and the policy attachment.

Without `depends_on`, Terraform might create the instance before the policy is attached, causing the startup script to fail with a permissions error.

Adding `depends_on = [aws_iam_role_policy_attachment.the_policy]` makes the dependency explicit — Terraform will wait for the policy attachment to complete before creating the instance.

- A is incorrect because Terraform only traces explicit attribute references; side effects from IAM role membership are not traceable through the graph.
- C addresses replacement ordering, not dependency ordering.
- D would suppress drift detection, which is unrelated to the ordering problem.

</details>

### Question 2

A team renames `resource "aws_s3_bucket" "old_bucket"` to `resource "aws_s3_bucket" "data_bucket"` in their Terraform configuration. What happens when they run `terraform plan` without any additional changes?

**A.** Terraform plans to destroy `old_bucket` and create `data_bucket` as a new resource.

**B.** Terraform detects the rename automatically and updates the state without any resource churn.

**C.** Terraform plans to destroy `old_bucket` and create `data_bucket`, but only if `prevent_destroy = true` is not set.

**D.** Terraform will error because renaming a resource is not supported.

<details><summary>Answer</summary>

**Answer: A**

Without a `moved` block, Terraform treats the renamed resource as a brand-new resource and the old address as a resource that should be destroyed. The plan will show:

```
- aws_s3_bucket.old_bucket   (destroy)
+ aws_s3_bucket.data_bucket  (create)
```

To rename without destroy/recreate, you must add:

```hcl
moved {
  from = aws_s3_bucket.old_bucket
  to   = aws_s3_bucket.data_bucket
}
```

- B is incorrect: Terraform does not detect renames — it only sees a missing resource and a new resource.
- C is incorrect: `prevent_destroy` would cause the plan to **error** if the bucket is being destroyed, not change the destroy/create behavior.
- D is incorrect: renaming is supported, but requires the `moved` block.

</details>

### Question 3

A production PostgreSQL RDS instance has `prevent_destroy = true` in its lifecycle block. A team member runs `terraform destroy`. What happens?

**A.** Terraform destroys the instance because `terraform destroy` overrides all lifecycle settings.

**B.** Terraform errors during plan with a message that the instance cannot be destroyed.

**C.** Terraform destroys the instance only if `-force` is passed.

**D.** Terraform skips the instance but destroys all other resources.

<details><summary>Answer</summary>

**Answer: B**

`prevent_destroy = true` causes Terraform to **error at plan time** whenever a plan would include destroying the resource. This applies to `terraform destroy`, `terraform apply` with a replacement plan, or any operation that would remove the resource.

The error message will be something like:

```
Error: Instance cannot be destroyed
  on main.tf line X:
  resource "aws_db_instance" "prod" has lifecycle.prevent_destroy set,
  but the plan calls for this resource to be destroyed.
```

To destroy the resource, the team must **remove the `prevent_destroy = true` line** from the configuration and run `terraform apply` first, then destroy.

- A is incorrect: `terraform destroy` does not override `prevent_destroy`.
- C is incorrect: there is no `-force` flag that overrides lifecycle settings.
- D is incorrect: `prevent_destroy` causes an error, not a skip — the entire plan fails.

</details>

---
## 12. Real-World Scenarios

<details>
<summary>Scenario 1: Zero-Downtime TLS Certificate Rotation with create_before_destroy</summary>

**Situation:**  
A team manages an AWS ALB listener that terminates TLS using an ACM certificate. The domain name changes from `api.example.com` to `api.v2.example.com`. Changing the domain on an ACM certificate requires **replacing** it (a new certificate is issued). By default, Terraform would destroy the old certificate before creating the new one — but the ALB listener is actively using it, causing a brief TLS error window.

**HCL Configuration:**

```hcl
resource "aws_acm_certificate" "api" {
  domain_name       = var.api_domain
  validation_method = "DNS"

  lifecycle {
    create_before_destroy = true
  }
}

resource "aws_lb_listener" "https" {
  load_balancer_arn = aws_lb.api.arn
  port              = 443
  protocol          = "HTTPS"
  certificate_arn   = aws_acm_certificate.api.arn

  default_action {
    type             = "forward"
    target_group_arn = aws_lb_target_group.api.arn
  }
}
```

**Expected Outcome:**  
Terraform creates the new certificate first, validates it, updates the listener to use the new ARN, then deletes the old certificate. Zero TLS errors.

**Exam Sub-Objective:** `create_before_destroy` lifecycle meta-argument — prevent destroy-before-create ordering.

</details>

<details>
<summary>Scenario 2: IAM Role Side Effect — Hidden Dependency with depends_on</summary>

**Situation:**  
An ECS task definition launches a container that immediately calls `aws s3 cp` in its entrypoint script. The ECS task role must have the S3 policy attached **before** the first task runs. The task definition references the task role ARN — but the policy attachment is a separate resource that does not feed any attribute into the task definition.

**HCL Configuration:**

```hcl
resource "aws_iam_role" "ecs_task" {
  name               = "ecs-task-role"
  assume_role_policy = data.aws_iam_policy_document.ecs_assume.json
}

resource "aws_iam_role_policy_attachment" "s3_write" {
  role       = aws_iam_role.ecs_task.name
  policy_arn = "arn:aws:iam::aws:policy/AmazonS3FullAccess"
}

resource "aws_ecs_task_definition" "worker" {
  family                   = "worker"
  task_role_arn            = aws_iam_role.ecs_task.arn
  network_mode             = "awsvpc"
  requires_compatibilities = ["FARGATE"]
  cpu                      = 256
  memory                   = 512

  container_definitions = jsonencode([{
    name  = "worker"
    image = "my-worker:latest"
    # entrypoint calls aws s3 cp — needs the policy to be attached
  }])

  # The policy attachment is a side effect — not visible in any attribute reference
  depends_on = [aws_iam_role_policy_attachment.s3_write]
}
```

**Expected Outcome:**  
The task definition is not created until the policy attachment is complete. Tasks launched from this definition will have S3 permissions from their first execution.

**Exam Sub-Objective:** Explicit `depends_on` for hidden (side-effect) dependencies.

</details>

<details>
<summary>Scenario 3: Multi-Region Deployment with Provider Aliases</summary>

**Situation:**  
A company needs to deploy identical S3 buckets in `us-east-1` (primary) and `eu-west-1` (GDPR-compliant EU region). Both buckets must be managed in the same Terraform workspace. The workspace uses the AWS provider twice — once per region — with aliases to distinguish them.

**HCL Configuration:**

```hcl
# ── providers.tf ─────────────────────────────────────────────────────────────
provider "aws" {
  region = "us-east-1"
}

provider "aws" {
  alias  = "eu"
  region = "eu-west-1"
}

# ── main.tf ──────────────────────────────────────────────────────────────────
resource "aws_s3_bucket" "primary" {
  bucket = "myapp-data-primary"
  # Uses default provider (us-east-1)
}

resource "aws_s3_bucket" "eu" {
  bucket   = "myapp-data-eu"
  provider = aws.eu   # Explicitly uses the aliased EU provider
}

# Both buckets are created in parallel since there's no dependency between them
```

**Expected Outcome:**  
Two S3 buckets are created — one in each region — in a single `terraform apply`. The EU bucket is created in `eu-west-1` because it explicitly selects the aliased provider.

**Exam Sub-Objective:** `provider` meta-argument with aliased provider configurations; multi-region deployments.

</details>

<details>
<summary>Scenario 4: Refactoring a Resource into a Module with the moved Block</summary>

**Situation:**  
A growing configuration has a VPC, subnets, and route tables defined in the root module. The team decides to extract all networking resources into a `./modules/networking` module. Without the `moved` block, running `terraform apply` after the refactor would destroy all existing network infrastructure and recreate it — causing a major outage.

**HCL Configuration:**

```hcl
# ── Before: resources were in root module ────────────────────────────────────
# resource "aws_vpc" "main"    { ... }
# resource "aws_subnet" "web"  { ... }
# resource "aws_subnet" "app"  { ... }

# ── After: resources moved into module ───────────────────────────────────────
module "networking" {
  source = "./modules/networking"
  cidr   = "10.0.0.0/16"
}

# ── moves.tf — in root module ─────────────────────────────────────────────────
moved {
  from = aws_vpc.main
  to   = module.networking.aws_vpc.main
}

moved {
  from = aws_subnet.web
  to   = module.networking.aws_subnet.web
}

moved {
  from = aws_subnet.app
  to   = module.networking.aws_subnet.app
}
```

**CLI Output (terraform plan):**

```
# aws_vpc.main has moved to module.networking.aws_vpc.main
  resource "aws_vpc" "main" {  ← No changes, just address update
  }

Plan: 0 to add, 0 to change, 0 to destroy.
```

**Expected Outcome:**  
No infrastructure changes. State is updated to reflect the new module addresses. The team can then remove the `moved` blocks after everyone has applied.

**Exam Sub-Objective:** `moved` block for refactoring — moving resources into modules without destroy/recreate.

</details>

<details>
<summary>Scenario 5: Protecting a Production Database and Safely Removing It from State</summary>

**Situation:**  
Phase 1: A production Aurora cluster has `prevent_destroy = true` to guard against accidents. A developer accidentally runs `terraform destroy`. Phase 2: Six months later, the database is handed off to another team's workspace. The current workspace needs to stop managing it without deleting it.

**Phase 1 — prevent_destroy protects the database:**

```hcl
resource "aws_rds_cluster" "aurora_prod" {
  cluster_identifier = "aurora-prod"
  engine             = "aurora-postgresql"
  database_name      = "appdb"
  master_username    = var.db_user
  master_password    = var.db_password

  lifecycle {
    prevent_destroy = true
  }
}
```

```bash
# Developer runs terraform destroy
$ terraform destroy

# Terraform errors immediately at plan time:
# Error: Instance cannot be destroyed
# resource "aws_rds_cluster" "aurora_prod" has lifecycle.prevent_destroy set
```

**Phase 2 — removed block hands off ownership:**

```hcl
# Remove the original resource block from config, add removed block instead:
removed {
  from = aws_rds_cluster.aurora_prod

  lifecycle {
    destroy = false   # Remove from state only — do NOT delete the database
  }
}
```

```bash
$ terraform apply
# Plan: 0 to add, 0 to change, 0 to destroy.
# State: aws_rds_cluster.aurora_prod removed from state.
# Real database: UNTOUCHED.
```

**Expected Outcome:**  
Phase 1: `terraform destroy` is blocked by `prevent_destroy`. Phase 2: The database remains running in AWS but is no longer tracked in this workspace's state. The other team's workspace can now import it.

**Exam Sub-Objective:** `prevent_destroy` guards against accidental deletion; `removed` block (destroy=false) for safe state removal without infrastructure impact.

</details>

---
## 13. Quick-Reference Cheat Sheet

### Dependency Types

| Type | How | When |
|---|---|---|
| **Implicit** | Reference `resource.type.name.attr` in another resource | Always preferred — zero configuration |
| **Explicit** | `depends_on = [resource.type.name]` | Side effects Terraform can't trace |

### Graph and Execution

| Concept | Key Fact |
|---|---|
| DAG | Directed Acyclic Graph — no cycles allowed |
| Parallel apply | Default 10 concurrent operations (`-parallelism=N`) |
| `terraform graph` | Outputs **DOT format** — needs Graphviz to render |
| `-target` | Apply one resource — useful for troubleshooting; can leave state inconsistent |

### Lifecycle Meta-Arguments

| Argument | Effect | ⭐ Exam Tip |
|---|---|---|
| `create_before_destroy = true` | New first, old destroyed second | Required for zero-downtime; propagates to dependents |
| `prevent_destroy = true` | Plan errors if destroy planned | Must **remove from config** to bypass; not a `-force` flag |
| `ignore_changes = [attr]` | Skip drift for specific attributes | Use `all` to ignore all changes |
| `replace_triggered_by = [ref]` | Replace this when reference changes | Useful for ASG + launch template patterns |

### Provider Aliases

```hcl
provider "aws" { alias = "west"; region = "us-west-2" }
resource "aws_s3_bucket" "west" { provider = aws.west }
```

- Default provider (no alias) is used automatically
- Aliased provider requires explicit `provider =` in each resource
- Pass aliases to modules via `providers = { aws.alias = aws.alias }`

### Refactoring Blocks

| Block | Infrastructure | State | Available Since |
|---|---|---|---|
| `moved` | Unchanged | Address updated | Terraform 1.1 |
| `removed` (destroy=false) | Unchanged | Entry removed | Terraform 1.7 |
| `removed` (destroy=true) | **Deleted** | Entry removed | Terraform 1.7 |

### Most Tested Patterns

```
⭐ depends_on vs implicit — know when each is needed
⭐ create_before_destroy — what propagates, when required
⭐ prevent_destroy — plan errors, must remove config to bypass
⭐ provider alias syntax — provider = aws.alias_name
⭐ moved block — no destroy/recreate on rename
⭐ -target — troubleshooting only, can leave state inconsistent
⭐ terraform graph — DOT format output
```